# 03 – Train mô hình phân loại ô đỗ

Train và so sánh các phương pháp trên crop từ `02_make_crops`:

| Mức | Phương pháp |
|---|---|
| 0 | Đoán lớp đa số |
| 1 | HOG + Linear SVM, LBP + RBF SVM (đặc trưng thủ công, giống hướng của bài báo PKLot gốc) |
| 2 | CNN nhỏ train từ đầu |
| Chính | **MobileNetV3-Small** (pretrained ImageNet) |
| So sánh | ResNet18 (pretrained ImageNet) |

Hai thí nghiệm:
1. **In-domain**: train trên tập train, chọn theo valid, báo cáo trên test (ngày mới, cùng camera) + tách theo camera.
2. **Leave-one-camera-out (LOCO)**: train trên 2 camera, test trên camera còn lại (camera chưa thấy).

Cuối notebook xuất `classifier.onnx` + `classifier.json` (thông số tiền xử lý) cho hệ thống chạy thật.

**Trên Kaggle:** *Add Input* → notebook **02_make_crops** (Kaggle nén output của nó thành `_output_.zip`,
notebook này tự giải nén ra ổ local, mất vài phút); *Accelerator* → **GPU T4**; *Internet* → **On**
(tải trọng số pretrained ImageNet). Chạy bằng *Save Version → Save & Run All*.

**Nếu lần chạy trước bị dừng giữa chừng:** *Add Input* thêm output của lần chạy đó → các mô hình đã có file `.pt`
được nạp lại, không train lại. Mỗi mô hình và `results.csv` được lưu ngay khi xong.

## 1. Cấu hình

In [ ]:
import json
import math
import os
import random
import shutil
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, precision_recall_fscore_support
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from torchvision import models
from tqdm.auto import tqdm

cv2.setNumThreads(1)          # tránh tranh chấp luồng với bộ đọc ảnh nhiều luồng bên dưới

SEED = 42
EPOCHS = 8                    # in-domain
LOCO_EPOCHS = 5               # leave-one-camera-out
BATCH_SIZE = 256
LR = 1e-3
VAL_SUBSAMPLE = 30000         # chỉ dùng một phần tập valid để chọn epoch cho nhanh
SVM_TRAIN_SAMPLES = 60000     # số crop train cho Linear SVM
RBF_TRAIN_SAMPLES = 15000     # RBF SVM chậm theo bình phương số mẫu
MODELS = ["small_cnn", "mobilenet_v3", "resnet18"]
LOCO_MODELS = ["hog_svm", "lbp_svm", "small_cnn", "mobilenet_v3"]
DEPLOY_MODEL = "mobilenet_v3"
PRETRAINED = "IMAGENET1K_V1"
RUN_BASELINES = True          # đặt False để bỏ qua HOG/LBP + SVM
RUN_LOCO = True               # đặt False để bỏ qua thí nghiệm leave-one-camera-out
NUM_THREADS = 2 * (os.cpu_count() or 2)   # luồng đọc ảnh (cv2.imread nhả GIL)
PREFETCH = 8                  # số batch đọc trước
HEARTBEAT_SECONDS = 300       # in tiến độ giữa epoch nếu một epoch quá lâu

# Hình học crop – phải khớp 02_make_crops và module chạy thật
SAVE_SIZE, SAVE_PAD = 160, 0.25     # crop lưu trên đĩa
INPUT_SIZE, RUNTIME_PAD = 128, 0.10  # CenterCrop(128) của ảnh 160 = bbox + 10%
C0 = (SAVE_SIZE - INPUT_SIZE) // 2
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
CLASSES = ["vacant", "occupied"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
OUT = WORK_DIR / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()

## 2. Dữ liệu

In [ ]:
import subprocess
import zipfile

EXTRACT_DIR = Path("/tmp/pklot_crops")   # ổ local: đọc nhanh hơn /kaggle/input, không tính vào output

def extract_zip_with_crops(zip_path: Path) -> Path | None:
    # Kaggle tự nén output nhiều file của notebook 02 thành _output_.zip → giải nén một lần ra EXTRACT_DIR
    with zipfile.ZipFile(zip_path) as zf:
        csv_name = next((n for n in zf.namelist() if n.endswith("crops.csv")), None)
    if csv_name is None:
        return None
    root = EXTRACT_DIR / Path(csv_name).parent
    if (EXTRACT_DIR / ".done").exists():
        return root
    print(f"Giải nén {zip_path} ({zip_path.stat().st_size / 1e9:.1f} GB) → {EXTRACT_DIR} ...")
    t0 = time.time()
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    ok = shutil.which("unzip") and subprocess.run(["unzip", "-q", "-o", str(zip_path), "-d", str(EXTRACT_DIR)]).returncode in (0, 1)
    if not ok:
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(EXTRACT_DIR)
    (EXTRACT_DIR / ".done").touch()
    print(f"Giải nén xong sau {time.time() - t0:.0f}s")
    return root

def find_crops_root() -> Path:
    for depth in range(0, 6):   # 1) crops.csv nằm thẳng trong input
        for csv in Path("/kaggle/input").glob("*/" * depth + "crops.csv"):
            return csv.parent
    for depth in range(0, 5):   # 2) output của notebook 02 bị Kaggle nén thành file zip
        for zip_path in Path("/kaggle/input").glob("*/" * depth + "*.zip"):
            root = extract_zip_with_crops(zip_path)
            if root:
                return root
    if (WORK_DIR / "crops" / "crops.csv").exists():
        return WORK_DIR / "crops"
    raise FileNotFoundError("Không tìm thấy crops.csv – hãy Add Input notebook 02_make_crops (hoặc dataset pklot-crops).")

CROPS_ROOT = find_crops_root()
crops = pd.read_csv(CROPS_ROOT / "crops.csv")
sample = cv2.imread(str(CROPS_ROOT / crops["path"].iloc[0]))
assert sample.shape[:2] == (SAVE_SIZE, SAVE_SIZE), f"Crop {sample.shape[:2]} ≠ {SAVE_SIZE} – chạy lại 02_make_crops bản mới"

train_df = crops[crops["split"] == "train"].reset_index(drop=True)
val_df = crops[crops["split"] == "valid"].reset_index(drop=True)
test_df = crops[crops["split"] == "test"].reset_index(drop=True)
val_small = val_df.sample(min(VAL_SUBSAMPLE, len(val_df)), random_state=SEED).reset_index(drop=True)
CAMERAS = sorted(crops["camera"].unique())

print("CROPS_ROOT =", CROPS_ROOT)
pd.crosstab([crops["split"], crops["camera"]], crops["cls"].map(dict(enumerate(CLASSES))), margins=True)

## 3. Đọc ảnh, tiền xử lý và augment

- **Đọc ảnh:** nhiều luồng trong cùng tiến trình (không dùng tiến trình con của DataLoader – trên Kaggle chúng có thể bị treo).
- **Augment trên GPU** cho cả batch, mỗi ảnh một bộ tham số riêng: cắt ngẫu nhiên (≈ nới 0–25% + dịch), xoay ±10°,
  lật ngang, đổi độ sáng / tương phản / bão hòa màu, làm mờ (20%), giả lập bóng đổ (30%).
- **Đánh giá / chạy thật:** `CenterCrop(128)` = bbox + 10%.

In [ ]:
class BatchLoader:
    """Trả về từng batch (ảnh uint8 RGB (N,3,160,160), nhãn). Đọc trước PREFETCH batch bằng nhiều luồng."""

    def __init__(self, df, shuffle=False, drop_last=False, batch_size=BATCH_SIZE):
        self.paths = [str(CROPS_ROOT / p) for p in df["path"]]
        self.labels = df["cls"].to_numpy()
        self.shuffle, self.drop_last, self.bs = shuffle, drop_last, batch_size

    def __len__(self):
        n = len(self.paths)
        return n // self.bs if self.drop_last and n >= self.bs else math.ceil(n / self.bs)

    def load(self, idx):
        imgs = np.stack([cv2.imread(self.paths[i])[:, :, ::-1] for i in idx])   # BGR → RGB
        return torch.from_numpy(np.ascontiguousarray(imgs)).permute(0, 3, 1, 2), torch.from_numpy(self.labels[idx])

    def __iter__(self):
        order = np.random.permutation(len(self.paths)) if self.shuffle else np.arange(len(self.paths))
        batches = [order[i:i + self.bs] for i in range(0, len(order), self.bs)][:len(self)]
        with ThreadPoolExecutor(NUM_THREADS) as pool:
            pending = deque(pool.submit(self.load, b) for b in batches[:PREFETCH])
            for b in batches[PREFETCH:]:
                yield pending.popleft().result()
                pending.append(pool.submit(self.load, b))
            while pending:
                yield pending.popleft().result()

MEAN_T, STD_T = torch.tensor(MEAN).view(1, 3, 1, 1), torch.tensor(STD).view(1, 3, 1, 1)

def normalize(x):
    return (x - MEAN_T.to(x.device)) / STD_T.to(x.device)

def gpu_eval_batch(x):
    # uint8 (N,3,160,160) → CenterCrop(128) = bbox + 10% → chuẩn hóa
    return normalize(x[:, :, C0:C0 + INPUT_SIZE, C0:C0 + INPUT_SIZE].float() / 255)

def uniform(n, lo, hi, device):
    return torch.empty(n, device=device).uniform_(lo, hi)

def gpu_train_batch(x):
    n, dev = x.shape[0], x.device
    x = x.float() / 255
    # 1) Cắt ngẫu nhiên (≈ RandomResizedCrop scale 0.44–1, tỷ lệ 0.85–1.18) + xoay ±10° + lật ngang trong một phép affine
    area = uniform(n, 0.44, 1.0, dev)
    ratio = torch.exp(uniform(n, math.log(0.85), math.log(1.18), dev))
    w, h = torch.sqrt(area * ratio).clamp(max=1.0), torch.sqrt(area / ratio).clamp(max=1.0)
    cx, cy = uniform(n, -1, 1, dev) * (1 - w), uniform(n, -1, 1, dev) * (1 - h)
    ang = uniform(n, -math.radians(10), math.radians(10), dev)
    flip = torch.where(torch.rand(n, device=dev) < 0.5, -1.0, 1.0)
    cos, sin = torch.cos(ang), torch.sin(ang)
    theta = torch.stack([torch.stack([w * cos * flip, -h * sin, cx], 1),
                         torch.stack([w * sin * flip, h * cos, cy], 1)], 1)
    grid = F.affine_grid(theta, (n, 3, INPUT_SIZE, INPUT_SIZE), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    # 2) Độ sáng, tương phản, bão hòa màu (≈ ColorJitter 0.4 / 0.4 / 0.3)
    x = x * uniform(n, 0.6, 1.4, dev).view(n, 1, 1, 1)
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    x = (x - mean) * uniform(n, 0.6, 1.4, dev).view(n, 1, 1, 1) + mean
    gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
    x = (x - gray) * uniform(n, 0.7, 1.3, dev).view(n, 1, 1, 1) + gray
    # 3) Làm mờ Gaussian 5×5 cho ~20% ảnh
    blur = torch.rand(n, device=dev) < 0.2
    if blur.any():
        sigma = random.uniform(0.5, 1.5)
        k = torch.exp(-(torch.arange(5, device=dev, dtype=x.dtype) - 2) ** 2 / (2 * sigma ** 2))
        k = k / k.sum()
        kernel = (k[:, None] * k[None, :]).expand(3, 1, 5, 5)
        x[blur] = F.conv2d(F.pad(x[blur], (2, 2, 2, 2), mode="reflect"), kernel, groups=3)
    # 4) Giả lập bóng đổ cho ~30% ảnh: làm tối một phía của đường thẳng ngẫu nhiên
    lin = torch.linspace(-0.5, 0.5, INPUT_SIZE, device=dev)
    ys, xs = torch.meshgrid(lin, lin, indexing="ij")
    a = uniform(n, 0, 2 * math.pi, dev).view(n, 1, 1)
    side = xs * torch.cos(a) + ys * torch.sin(a) > uniform(n, -0.3, 0.3, dev).view(n, 1, 1)
    shadow = (torch.rand(n, device=dev) < 0.3).view(n, 1, 1) & side
    factor = uniform(n, 0.4, 0.8, dev).view(n, 1, 1)
    x = x * torch.where(shadow, factor, torch.ones_like(factor)).unsqueeze(1)
    return normalize(x.clamp(0, 1))

def denorm(x):
    return (x.permute(1, 2, 0) * torch.tensor(STD) + torch.tensor(MEAN)).clamp(0, 1).numpy()

# Xem thử: cột đầu là ảnh dùng khi đánh giá, các cột sau là ảnh augment khi train
samples = train_df.groupby("cls").sample(2, random_state=SEED)
x, _ = BatchLoader(samples).load(np.arange(len(samples)))
with torch.no_grad():
    shown_eval = gpu_eval_batch(x.to(DEVICE)).cpu()
    shown_aug = [gpu_train_batch(x.to(DEVICE)).cpu() for _ in range(6)]
fig, axes = plt.subplots(len(samples), 7, figsize=(13, 7.6))
for r, (_, row) in enumerate(samples.iterrows()):
    axes[r][0].imshow(denorm(shown_eval[r])); axes[r][0].set_title(f"eval – {CLASSES[row['cls']]}", fontsize=9)
    for j in range(6):
        axes[r][j + 1].imshow(denorm(shown_aug[j][r]))
    for ax in axes[r]:
        ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "augmentation_samples.png", dpi=110)
plt.show()

## 4. Mô hình, train và đánh giá

In [ ]:
class SmallCNN(nn.Module):
    # Baseline deep learning train từ đầu (cỡ tương tự mAlexNet)
    def __init__(self):
        super().__init__()
        def block(cin, cout):
            return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout),
                                 nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, 2))

    def forward(self, x):
        return self.head(self.features(x))

def build_model(name):
    try:
        if name == "small_cnn":
            return SmallCNN()
        if name == "mobilenet_v3":
            m = models.mobilenet_v3_small(weights=PRETRAINED)
            m.classifier[3] = nn.Linear(m.classifier[3].in_features, 2)
            return m
        if name == "resnet18":
            m = models.resnet18(weights=PRETRAINED)
            m.fc = nn.Linear(m.fc.in_features, 2)
            return m
    except Exception as e:
        raise RuntimeError("Không tải được trọng số pretrained – bật Settings → Internet trên Kaggle.") from e
    raise ValueError(name)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

def metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred), "precision": p, "recall": r, "f1": f1}

def find_previous(filename):
    # File của lần chạy trước (output được Add Input) → dùng lại thay vì train lại
    for depth in range(1, 8):
        for p in Path("/kaggle/input").glob("*/" * depth + filename):
            return p
    return None

@torch.no_grad()
def predict(model, df):
    # Trả về xác suất occupied cho từng crop
    model.eval()
    probs = []
    for x, _ in BatchLoader(df):
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            logits = model(gpu_eval_batch(x.to(DEVICE, non_blocking=True)))
        probs.append(torch.softmax(logits.float(), 1)[:, 1].cpu())
    return torch.cat(probs).numpy()

def train_model(name, train_part, val_part, epochs, tag, file_tag):
    seed_everything()
    model = build_model(name).to(DEVICE)
    previous = find_previous(f"{file_tag}.pt")
    if previous:
        print(f"{tag}: dùng lại trọng số {previous}")
        model.load_state_dict(torch.load(previous, map_location=DEVICE))
        shutil.copy(previous, OUT / f"{file_tag}.pt")
        hist_path = find_previous(f"history_{file_tag}.csv")
        if hist_path:
            shutil.copy(hist_path, OUT / hist_path.name)
        return model, pd.read_csv(hist_path) if hist_path else pd.DataFrame()

    loader = BatchLoader(train_part, shuffle=True, drop_last=len(train_part) > BATCH_SIZE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=epochs * len(loader), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")
    loss_fn = nn.CrossEntropyLoss()
    best_f1, best_state, history = -1.0, None, []
    for epoch in range(1, epochs + 1):
        model.train()
        t0 = last = time.time()
        total, n = 0.0, 0
        for i, (x, y) in enumerate(loader, 1):
            x, y = gpu_train_batch(x.to(DEVICE, non_blocking=True)), y.to(DEVICE, non_blocking=True)
            with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
                loss = loss_fn(model(x), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            total, n = total + loss.item() * len(y), n + len(y)
            if time.time() - last > HEARTBEAT_SECONDS:
                print(f"  {tag} epoch {epoch}: {i}/{len(loader)} batch, loss {total / n:.4f}", flush=True)
                last = time.time()
        val = metrics(val_part["cls"].to_numpy(), (predict(model, val_part) >= 0.5).astype(int))
        history.append({"epoch": epoch, "train_loss": total / n, **{f"val_{k}": v for k, v in val.items()}})
        print(f"{tag} epoch {epoch}/{epochs}: loss {total / n:.4f} | val acc {val['accuracy']:.4f} f1 {val['f1']:.4f} | "
              f"{time.time() - t0:.0f}s", flush=True)
        if val["f1"] > best_f1:
            best_f1 = val["f1"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    torch.save(best_state, OUT / f"{file_tag}.pt")            # lưu ngay, phòng khi notebook dừng giữa chừng
    history = pd.DataFrame(history)
    history.to_csv(OUT / f"history_{file_tag}.csv", index=False)
    return model, history

RESULTS = []

def record(experiment, model_name, df, y_pred, params=None):
    # Lưu metric tổng và theo từng camera của tập test; ghi results.csv ngay sau mỗi lần
    y_true, y_pred = df["cls"].to_numpy(), np.asarray(y_pred).astype(int)
    cams = sorted(df["camera"].unique())
    groups = ([("all", np.ones(len(df), bool))] if len(cams) > 1 else []) + [(c, (df["camera"] == c).to_numpy()) for c in cams]
    for cam, mask in groups:
        RESULTS.append({"experiment": experiment, "model": model_name, "test_camera": cam, "n": int(mask.sum()),
                        **metrics(y_true[mask], y_pred[mask]), "params": params})
    pd.DataFrame(RESULTS).to_csv(OUT / "results.csv", index=False)

## 5. Baseline: lớp đa số và đặc trưng thủ công + SVM

Đặc trưng tính trên ảnh xám 64×64 của vùng `CenterCrop(128)`:
- **HOG** (cell 8×8, block 2×2, 9 hướng – 1.764 chiều) → Linear SVM.
- **LBP uniform** 8 lân cận, lưới 2×2 (236 chiều) → RBF SVM.

Lưu ý khi so sánh: SVM chỉ train trên tập con (`SVM_TRAIN_SAMPLES`, RBF `RBF_TRAIN_SAMPLES`) vì chi phí tính toán.

In [ ]:
LBP_OFFSETS = [(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]   # theo vòng tròn

def _transitions(v):
    bits = [(v >> i) & 1 for i in range(8)]
    return sum(bits[i] != bits[(i + 1) % 8] for i in range(8))

UNIFORM = [v for v in range(256) if _transitions(v) <= 2]            # 58 mẫu uniform
LBP_LUT = np.full(256, len(UNIFORM), np.uint8)                       # mẫu không uniform → nhãn 58
LBP_LUT[UNIFORM] = np.arange(len(UNIFORM))

def load_gray(path):
    img = cv2.imread(str(CROPS_ROOT / path), cv2.IMREAD_GRAYSCALE)[C0:C0 + INPUT_SIZE, C0:C0 + INPUT_SIZE]
    return cv2.resize(img, (64, 64), interpolation=cv2.INTER_AREA)

def hog_feat(g, cell=8, bins=9):
    # HOG cài bằng NumPy (không phụ thuộc phiên bản OpenCV): cell 8×8, block 2×2, chuẩn hóa L2-Hys
    gx = cv2.Sobel(g.astype(np.float32), cv2.CV_32F, 1, 0, ksize=1)
    gy = cv2.Sobel(g.astype(np.float32), cv2.CV_32F, 0, 1, ksize=1)
    mag, ang = cv2.cartToPolar(gx, gy, angleInDegrees=True)
    b = np.minimum(((ang % 180) / (180 / bins)).astype(int), bins - 1)
    h, w = g.shape
    rows, cols = np.indices((h, w))
    hist = np.zeros((h // cell, w // cell, bins), np.float32)
    np.add.at(hist, (rows // cell, cols // cell, b), mag)
    blocks = np.concatenate([hist[:-1, :-1], hist[1:, :-1], hist[:-1, 1:], hist[1:, 1:]], axis=2)
    for _ in range(2):   # L2 → cắt 0.2 → L2
        blocks = blocks / (np.linalg.norm(blocks, axis=2, keepdims=True) + 1e-6)
        blocks = np.minimum(blocks, 0.2) if _ == 0 else blocks
    return blocks.ravel()

def lbp_feat(g):
    g = g.astype(np.int16)
    c = g[1:-1, 1:-1]
    h, w = c.shape
    code = np.zeros(c.shape, np.uint8)
    for k, (dy, dx) in enumerate(LBP_OFFSETS):
        code |= (g[1 + dy:1 + dy + h, 1 + dx:1 + dx + w] >= c).astype(np.uint8) << k
    labels = LBP_LUT[code]
    hists = [np.bincount(cell.ravel(), minlength=len(UNIFORM) + 1)
             for rows in np.array_split(labels, 2, 0) for cell in np.array_split(rows, 2, 1)]
    return np.concatenate([hh / hh.sum() for hh in hists]).astype(np.float32)

def extract(df):
    def both(path):
        g = load_gray(path)
        return hog_feat(g), lbp_feat(g)
    with ThreadPoolExecutor(NUM_THREADS) as pool:
        out = list(tqdm(pool.map(both, df["path"], chunksize=256), total=len(df), desc="features", mininterval=60))
    return {"hog_svm": np.stack([o[0] for o in out]), "lbp_svm": np.stack([o[1] for o in out])}

def fit_svm(method, X, y):
    if method == "hog_svm":
        clf = make_pipeline(StandardScaler(), LinearSVC(C=0.1, dual=False))
    else:
        X, y = X[:RBF_TRAIN_SAMPLES], y[:RBF_TRAIN_SAMPLES]
        clf = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=10, gamma="scale"))
    return clf.fit(X, y)

majority = int(train_df["cls"].mode()[0])
record("in-domain", "majority", test_df, np.full(len(test_df), majority), params=0)
if RUN_BASELINES:
    svm_train_df = train_df.sample(min(SVM_TRAIN_SAMPLES, len(train_df)), random_state=SEED).reset_index(drop=True)
    t0 = time.time()
    F_train, F_test = extract(svm_train_df), extract(test_df)
    print(f"Trích đặc trưng: {time.time() - t0:.0f}s")
    for method in ["hog_svm", "lbp_svm"]:
        t0 = time.time()
        clf = fit_svm(method, F_train[method], svm_train_df["cls"].to_numpy())
        record("in-domain", method, test_df, clf.predict(F_test[method]))
        print(f"{method}: {time.time() - t0:.0f}s, acc {RESULTS[-4]['accuracy']:.4f}", flush=True)
pd.DataFrame(RESULTS).query("test_camera == 'all'")

## 6. Thí nghiệm in-domain: CNN

In [ ]:
trained, histories, test_probs = {}, {}, {}
for name in MODELS:
    model, hist = train_model(name, train_df, val_small, EPOCHS, tag=name, file_tag=name)
    trained[name], histories[name] = model, hist
    test_probs[name] = predict(model, test_df)
    record("in-domain", name, test_df, test_probs[name] >= 0.5, params=count_params(model))
    print(f"==> {name}: test acc {RESULTS[-4]['accuracy']:.4f}", flush=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, hist in histories.items():
    if len(hist):
        axes[0].plot(hist["epoch"], hist["train_loss"], marker="o", label=name)
        axes[1].plot(hist["epoch"], hist["val_f1"], marker="o", label=name)
axes[0].set_title("Train loss"); axes[1].set_title("Val F1")
for ax in axes:
    ax.set_xlabel("epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT / "training_curves.png", dpi=120)
plt.show()

## 7. Thí nghiệm leave-one-camera-out

Mỗi lượt bỏ hẳn một camera khỏi train/valid và chỉ test trên camera đó → đo khả năng chạy trên camera chưa thấy.

In [ ]:
if RUN_LOCO:
    for cam in CAMERAS:
        tr = train_df[train_df["camera"] != cam]
        va = val_small[val_small["camera"] != cam]
        te_mask = (test_df["camera"] == cam).to_numpy()
        te = test_df[te_mask]
        for name in LOCO_MODELS:
            if name.endswith("_svm"):
                if not RUN_BASELINES:
                    continue
                svm_mask = (svm_train_df["camera"] != cam).to_numpy()
                clf = fit_svm(name, F_train[name][svm_mask], svm_train_df["cls"].to_numpy()[svm_mask])
                pred, params = clf.predict(F_test[name][te_mask]), None
            else:
                model, _ = train_model(name, tr, va, LOCO_EPOCHS, tag=f"LOCO-{cam} {name}", file_tag=f"loco_{cam}_{name}")
                pred, params = predict(model, te) >= 0.5, count_params(model)
                del model
            record("LOCO", name, te, pred, params=params)
            print(f"LOCO bỏ {cam} – {name}: acc {RESULTS[-1]['accuracy']:.4f}", flush=True)

## 8. Tổng hợp kết quả

In [ ]:
res = pd.DataFrame(RESULTS)
res.to_csv(OUT / "results.csv", index=False)
order = ["majority", "hog_svm", "lbp_svm", "small_cnn", "mobilenet_v3", "resnet18"]

print("In-domain (test PKLot, mọi camera):")
display(res.query("experiment == 'in-domain' and test_camera == 'all'")
        .set_index("model").reindex([m for m in order if m in set(res["model"])])
        [["accuracy", "precision", "recall", "f1", "params"]].round(4))

per_cam = res.query("experiment == 'in-domain' and test_camera != 'all'").pivot(index="model", columns="test_camera", values="accuracy")
print("Accuracy theo camera – in-domain:"); display(per_cam.round(4))
loco = res.query("experiment == 'LOCO'").pivot(index="model", columns="test_camera", values="accuracy")
if len(loco):
    loco["mean"] = loco.mean(axis=1)
    print("Accuracy theo camera bị bỏ ra – LOCO:"); display(loco.round(4))

    # Mức giảm khi sang camera chưa thấy
    shared = [m for m in order if m in loco.index and m in per_cam.index]
    fig, axes = plt.subplots(1, len(CAMERAS), figsize=(5 * len(CAMERAS), 4), sharey=True)
    for ax, cam in zip(np.atleast_1d(axes), CAMERAS):
        x = np.arange(len(shared))
        ax.bar(x - 0.2, per_cam.loc[shared, cam], 0.4, label="in-domain")
        ax.bar(x + 0.2, loco.loc[shared, cam], 0.4, label="camera chưa thấy (LOCO)")
        ax.set_xticks(x, shared, rotation=20); ax.set_title(cam); ax.grid(axis="y", alpha=0.3)
    np.atleast_1d(axes)[0].set_ylabel("accuracy"); np.atleast_1d(axes)[0].legend()
    plt.tight_layout()
    plt.savefig(OUT / "loco_vs_indomain.png", dpi=120)
    plt.show()

## 9. Phân tích lỗi của mô hình triển khai

In [ ]:
probs = test_probs[DEPLOY_MODEL]
pred = (probs >= 0.5).astype(int)
y = test_df["cls"].to_numpy()
ConfusionMatrixDisplay(confusion_matrix(y, pred), display_labels=CLASSES).plot(cmap="Blues", values_format="d")
plt.title(f"{DEPLOY_MODEL} – test PKLot")
plt.savefig(OUT / "confusion_matrix.png", dpi=120)
plt.show()

# Các lỗi mô hình tự tin nhất → đưa vào phần phân tích lỗi của báo cáo
wrong = np.where(pred != y)[0]
wrong = wrong[np.argsort(-np.abs(probs[wrong] - 0.5))][:24]
if len(wrong):
    fig, axes = plt.subplots(3, 8, figsize=(16, 6.6))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), wrong):
        img = cv2.imread(str(CROPS_ROOT / test_df["path"][i]))[C0:C0 + INPUT_SIZE, C0:C0 + INPUT_SIZE]
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"thật: {CLASSES[y[i]]}\nP(occ)={probs[i]:.2f} {test_df['camera'][i]}", fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT / "errors.png", dpi=110)
    plt.show()
print(f"Số lỗi: {len(np.where(pred != y)[0]):,} / {len(y):,}")

## 10. Xuất ONNX cho hệ thống chạy thật

`classifier.json` ghi lại tiền xử lý mà module `src/vision` phải làm giống hệt:
cắt hình chữ nhật bao ô, nới `runtime_pad` mỗi phía → resize `input_size` → chuẩn hóa `mean/std` → softmax, lớp 1 = occupied.

In [ ]:
deploy = trained[DEPLOY_MODEL].eval().cpu()
onnx_path = OUT / "classifier.onnx"
torch.onnx.export(deploy, torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE), onnx_path,
                  input_names=["input"], output_names=["logits"],
                  dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}}, opset_version=17, dynamo=False)

summary = res.query("experiment == 'in-domain' and model == @DEPLOY_MODEL and test_camera == 'all'").iloc[0]
config = {
    "model": DEPLOY_MODEL, "classes": CLASSES, "input_size": INPUT_SIZE, "runtime_pad": RUNTIME_PAD,
    "mean": MEAN, "std": STD, "color": "RGB", "input_name": "input", "output_name": "logits",
    "test_accuracy": round(float(summary["accuracy"]), 5), "test_f1": round(float(summary["f1"]), 5),
}
(OUT / "classifier.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))
print(f"Đã lưu {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)")

# Kiểm tra ONNX Runtime cho cùng kết quả với PyTorch
try:
    import onnxruntime as ort
except ImportError:
    print("Bỏ qua kiểm tra: môi trường chưa có onnxruntime (sẽ kiểm tra trên máy local).")
else:
    x, _ = BatchLoader(test_df.head(64)).load(np.arange(min(64, len(test_df))))
    x = gpu_eval_batch(x)
    with torch.no_grad():
        ref = deploy(x).numpy()
    got = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"]).run(None, {"input": x.numpy()})[0]
    diff = float(np.abs(ref - got).max())
    print(f"Chênh lệch lớn nhất PyTorch vs ONNX: {diff:.2e}")
    assert diff < 1e-3
print(json.dumps(config, indent=2, ensure_ascii=False))

## Kết quả cần tải về
Tab *Output* → `outputs/`:
- `classifier.onnx`, `classifier.json` → thư mục `smart-parking/models/` trên máy.
- `results.csv` và các hình `*.png` → dùng cho chương Thực nghiệm.
- `*.pt` (trọng số PyTorch) → dùng lại khi fine-tune trên camera demo (`05_finetune_demo`).